# Gaze Direction Estimation — YOLO11 + L2CS-Net / UniGaze

| Stage | Model | Output |
|---|---|---|
| Face detection | `YoloFace11Detector` | bounding box + face crop |
| Gaze (ResNet-50) | `L2csNetWrapper` | (yaw, pitch) radians, 448×448 input |
| Gaze (ViT) | `UnigazeWrapper` | (yaw, pitch) radians, 224×224 input |

Both wrappers share the same API:
- `model.preprocess(frames)` → normalised float tensor on device
- `model.inference(tensor)` → `(yaw, pitch)` tensors in radians
- `model(frames)` → shorthand for preprocess + inference
- `model.predict(frames, roll_angles)` → optional on-device roll correction
- `GazeWrapper.visualize(faces, yaw, pitch)` → annotated crops with gaze arrow
- `GazeWrapper.looking_at_camera(yaw, pitch, thr)` → boolean tensor

In [ ]:
import math

import matplotlib.pyplot as plt

from exordium import FIXTURE_DIR
from exordium.video.core.io import image_to_np

face_path = FIXTURE_DIR / "image" / "face.jpg"
face_rgb = image_to_np(face_path, channel_order="RGB")  # (H, W, 3) uint8

plt.figure(figsize=(5, 5))
plt.imshow(face_rgb)
plt.title("face.jpg — fixture image")
plt.axis("off")
plt.tight_layout()
plt.show()

## Load models

In [ ]:
from exordium.video.face.detector.yolo11 import YoloFace11Detector
from exordium.video.face.gaze import L2csNetWrapper, UnigazeWrapper
from exordium.video.face.gaze.base import GazeWrapper

detector = YoloFace11Detector(device_id=None, conf=0.7)
l2cs = L2csNetWrapper(device_id=None)
unigaze = UnigazeWrapper(device_id=None)

## Step 1 — Face detection and crop

In [ ]:
dets = detector.detect_image(face_rgb)
det = max(dets, key=lambda d: d.score)
face_crop = det.crop(square=True, extra_space=1.3)  # (3, H', W') uint8 RGB tensor
face_tensor = face_crop.unsqueeze(0)  # (1, 3, H', W') for batched API

print(f"Detection: score={det.score:.3f}")
print(f"face_crop  : {tuple(face_crop.shape)}")
print(f"face_tensor: {tuple(face_tensor.shape)}  dtype={face_tensor.dtype}")

plt.figure(figsize=(4, 4))
plt.imshow(face_crop.permute(1, 2, 0).numpy())
plt.title("face crop")
plt.axis("off")
plt.tight_layout()
plt.show()

## Step 2 — L2CS-Net step by step

`preprocess` resizes to 448×448 and applies ImageNet normalisation.
`inference` runs ResNet-50 with softmax over 90 bins and returns angles in radians.

In [ ]:
tensor_448 = l2cs.preprocess(face_tensor)  # (1, 3, 448, 448) float32
yaw_l, pitch_l = l2cs.inference(tensor_448)  # (1,) radians each

print(f"Preprocessed: {tuple(tensor_448.shape)}  dtype={tensor_448.dtype}")
print("L2CS-Net:")
print(f"  yaw   = {yaw_l[0]:.4f} rad  ({math.degrees(yaw_l[0].item()):+.1f}°)")
print(f"  pitch = {pitch_l[0]:.4f} rad  ({math.degrees(pitch_l[0].item()):+.1f}°)")

# Shorthand: __call__ chains preprocess + inference
yaw_l, pitch_l = l2cs(face_tensor)
print(f"\n__call__ output: yaw={tuple(yaw_l.shape)}  pitch={tuple(pitch_l.shape)}")

## Step 3 — UniGaze step by step

`preprocess` resizes to 224×224 and applies ImageNet normalisation.
`inference` runs a ViT and extracts `pred_gaze` from the output dict.

In [ ]:
tensor_224 = unigaze.preprocess(face_tensor)  # (1, 3, 224, 224) float32
yaw_u, pitch_u = unigaze.inference(tensor_224)  # (1,) radians each

print(f"Preprocessed: {tuple(tensor_224.shape)}  dtype={tensor_224.dtype}")
print("UniGaze:")
print(f"  yaw   = {yaw_u[0]:.4f} rad  ({math.degrees(yaw_u[0].item()):+.1f}°)")
print(f"  pitch = {pitch_u[0]:.4f} rad  ({math.degrees(pitch_u[0].item()):+.1f}°)")

## Step 4 — Gaze vector visualisation

`GazeWrapper.visualize` draws a red arrow for gaze direction, a red outer circle
for maximum range, and a green inner circle for the looking-at-camera threshold.

In [ ]:
def show(vis_list, ax, title):
    img = vis_list[0]
    if hasattr(img, "permute"):
        img = img.permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")


vis_l2cs = GazeWrapper.visualize(face_tensor, yaw_l, pitch_l)
vis_unigaze = GazeWrapper.visualize(face_tensor, yaw_u, pitch_u)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
show(
    vis_l2cs,
    axes[0],
    f"L2CS-Net\nyaw={math.degrees(yaw_l[0].item()):+.1f}°  pitch={math.degrees(pitch_l[0].item()):+.1f}°",
)
show(
    vis_unigaze,
    axes[1],
    f"UniGaze\nyaw={math.degrees(yaw_u[0].item()):+.1f}°  pitch={math.degrees(pitch_u[0].item()):+.1f}°",
)
plt.suptitle("Step 4 — gaze vector visualisation", fontsize=13)
plt.tight_layout()
plt.show()

## Step 5 — Looking-at-camera detection

In [ ]:
looking_l2cs = GazeWrapper.looking_at_camera(yaw_l, pitch_l, thr=0.3)
looking_unigaze = GazeWrapper.looking_at_camera(yaw_u, pitch_u, thr=0.3)

print(f"L2CS-Net  looking at camera: {looking_l2cs[0].item()}")
print(f"UniGaze   looking at camera: {looking_unigaze[0].item()}")
print(f"(threshold = 0.3 rad ≈ {math.degrees(0.3):.1f}°)")

## Step 6 — Roll correction

`predict(frames, roll_angles)` rotates each face by `-roll` on-device before
inference, correcting for head tilt without a numpy round-trip.

In [ ]:
roll_deg = 15.0
yaw_r, pitch_r = l2cs.predict(face_tensor, roll_angles=[roll_deg])

print(
    f"Without roll correction:  yaw={math.degrees(yaw_l[0].item()):+.1f}°  pitch={math.degrees(pitch_l[0].item()):+.1f}°"
)
print(
    f"With roll={roll_deg}° correction:  yaw={math.degrees(yaw_r[0].item()):+.1f}°  pitch={math.degrees(pitch_r[0].item()):+.1f}°"
)

## Step 7 — Batch processing

In [ ]:
batch = face_tensor.expand(3, -1, -1, -1).clone()  # (3, 3, H', W') uint8

yaw_b, pitch_b = l2cs(batch)  # (3,) each
looking_b = GazeWrapper.looking_at_camera(yaw_b, pitch_b, thr=0.3)

print(f"Batch input : {tuple(batch.shape)}")
for i in range(len(yaw_b)):
    print(
        f"  face {i}: yaw={math.degrees(yaw_b[i].item()):+.1f}°  "
        f"pitch={math.degrees(pitch_b[i].item()):+.1f}°  "
        f"looking_at_camera={looking_b[i].item()}"
    )